# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maahilkureshi-design/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane - or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane: Refresh / Content Opportunity Scoring.** I chose it because the decision is concrete (which page a human reviews first), the target can be observed in a later time window instead of being defined by a rule, and the lane rewards careful time-aware validation and ranking evaluation, which I have used before in my MSc thesis work on recommender systems. The lane is now locked; the data checks so far (see the data contract) gave me no reason to change it.

In [1]:
LANE = 'Refresh / Content Opportunity Scoring'
print('Lane:', LANE)

Lane: Refresh / Content Opportunity Scoring


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Decision.** Which pages should a content reviewer look at first, out of thousands, because their search visibility is likely to fall?

**Who acts, and how.** A content reviewer works a ranked queue each cycle and has time for roughly 20 to 50 pages. For each page they choose an action: refresh or improve, protect, merge or prune, or monitor.

**Cost of a wrong call.** With fixed reviewer time, a false alarm wastes a review slot and a missed decline stays unnoticed longer. I treat wasted slots as the main cost and will report recall@K so missed declines stay visible.

**Why more than a rule.** The product already flags pages with hand-written thresholds. Those flags are outputs, so I use them only as a baseline to beat, never as features. The open question is whether a model trained on past-to-future outcomes ranks pages better than a transparent rule on the same held-out period. It may not, and that would be a finding too.

In [2]:
REVIEW_CAPACITY = [20, 50]  # pages a reviewer can act on per cycle
print('Review capacity assumptions (pages per cycle):', REVIEW_CAPACITY)

Review capacity assumptions (pages per cycle): [20, 50]


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

Numbers from the starter sample (the cell below recomputes them):
- 30,000 pages from 32 clients. 54.2% carry a "down" trend flag (last 30 days vs the 30 days before).
- Among pages with at least 100 impressions in the prior 30 days, 61.6% are flagged down.
- The top 10% of declining pages (1,626 pages) hold 68.7% of all impressions lost by declining pages.

The last number is the case for ranking: a small share of pages carries most of the loss, so order matters. Caution: this is a starter sample with a current-window trend flag, not the capstone target, and the high "down" share may reflect how the sample was built. These numbers are observed and directional.

In [3]:
import pandas as pd

STARTER_LOCAL = '../../data/raw/content_refresh_anonymized.csv'
STARTER_URL = 'https://raw.githubusercontent.com/maahilkureshi-design/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv'
try:
    df = pd.read_csv(STARTER_LOCAL)
except FileNotFoundError:
    df = pd.read_csv(STARTER_URL)

n_pages = len(df)
n_clients = df['client_id'].nunique()
share_down = (df['trend_direction'] == 'down').mean()

eligible = df[df['impressions_prev_30d'] >= 100]
share_down_eligible = (eligible['trend_direction'] == 'down').mean()

down = df[df['trend_direction'] == 'down'].copy()
down['lost'] = (down['impressions_prev_30d'] - down['impressions_last_30d']).clip(lower=0)
top_n = int(len(down) * 0.10)
share_of_loss_top10 = down['lost'].nlargest(top_n).sum() / down['lost'].sum()

print(f"Pages: {n_pages:,} from {n_clients} clients")
print(f"Flagged down: {share_down:.1%} of all pages; {share_down_eligible:.1%} of pages with >=100 prior-30d impressions (n={len(eligible):,})")
print(f"Top 10% of declining pages (n={top_n:,}) hold {share_of_loss_top10:.1%} of lost impressions")

Pages: 30,000 from 32 clients
Flagged down: 54.2% of all pages; 61.6% of pages with >=100 prior-30d impressions (n=18,010)
Top 10% of declining pages (n=1,626) hold 68.7% of lost impressions


### 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) - and what it never will (causal proof, 'predicting Google').*

**I can say:** in this data we observed patterns; certain signals are associated with later declines; a validated model ranks pages at a stated precision@K against a baseline on a held-out period; the queue is decision-support for where a reviewer should look first.

**I will not say:** that a refresh caused a recovery (no experiment was run), anything about how Google ranks pages, or that a flagged page will certainly decline. Every headline number comes with its base rate and sample size.

In [4]:
CAN_SAY = ['observed', 'associated with', 'ranks at precision@K vs baseline', 'decision-support']
CANNOT_SAY = ['causes / proves', "Google's algorithm", 'guaranteed decline or recovery']
print('Can say:', CAN_SAY)
print('Cannot say:', CANNOT_SAY)

Can say: ['observed', 'associated with', 'ranks at precision@K vs baseline', 'decision-support']
Cannot say: ['causes / proves', "Google's algorithm", 'guaranteed decline or recovery']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.